In [ ]:
"""
Reinforcement learning model implementation
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from dataclasses import dataclass
from typing import Tuple, Optional
from config import settings


@dataclass
class RLModelResult:
    """Container for RL model results."""
    name: str
    train_loss_list: list
    test_loss_list: list
    Q_pred: torch.Tensor
    A_batch: torch.Tensor
    mask_batch: torch.Tensor
    R_batch: Optional[torch.Tensor] = None


class BiLSTMAttentionRL(nn.Module):
    """BiLSTM with Attention for RL."""
    
    def __init__(self, input_dim: int, num_actions: int, hidden_dim: int = 64, 
                 num_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.rnn = nn.LSTM(
            input_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout
        )
        self.attn_fc = nn.Linear(hidden_dim * 2, 1)
        self.fc_out = nn.Linear(hidden_dim * 2, num_actions)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass."""
        out, _ = self.rnn(x)
        attn_scores = torch.softmax(self.attn_fc(out), dim=1)
        out = out * attn_scores
        out = self.dropout(out)
        Q = self.fc_out(out)
        return Q, attn_scores


def prepare_rl_data(df: pd.DataFrame, target_state: np.ndarray = None, alpha: float = None):
    """Prepare data for reinforcement learning."""
    if target_state is None:
        target_state = settings.TARGET_STATE
    if alpha is None:
        alpha = settings.ALPHA
    
    # Define columns
    id_cols = ['ID', 'ID2', 'Time']
    state_cols = settings.LAB_TESTS
    action_cols = ['Treat_Other', 'Treat_Anticoagulant', 'Treat_Antiplatelet', 
                   'Treat_Hemostatic', 'Treat_Blood Product']
    score_cols = ['Y_x', 'Y_y']
    reward_cols = ['reward_smooth']
    
    # Fill missing values and create masks
    df_filled = df.copy()
    df_filled[state_cols] = df_filled[state_cols].fillna(0.0)
    
    for col in state_cols:
        df_filled[col + "_mask"] = (~df[col].isna()).astype(float)
    
    state_cols_with_mask = state_cols + [c + "_mask" for c in state_cols]
    
    # Create final RL table
    df_rl = df_filled[id_cols + state_cols_with_mask + action_cols + 
                     score_cols + reward_cols].copy()
    
    # Calculate shaped reward
    states = df_rl[score_cols].fillna(0.0).values
    dist_to_target = np.linalg.norm(states - target_state, axis=1)
    df_rl['reward_shaped'] = df_rl['reward_smooth'] * 10 + alpha * (-dist_to_target)
    
    # Sort by patient and time
    df_rl = df_rl.sort_values(['ID', 'ID2', 'Time']).reset_index(drop=True)
    
    return df_rl


def train_rl_model(df_rl: pd.DataFrame, device: str = None) -> RLModelResult:
    """Train the RL model."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Prepare data
    ids = df_rl['ID'].unique()
    B = len(ids)
    T = settings.MAX_STEPS
    
    state_cols = settings.LAB_TESTS + ['Y_x', 'Y_y']
    action_cols = ['Treat_Other', 'Treat_Anticoagulant', 'Treat_Antiplatelet', 
                   'Treat_Hemostatic', 'Treat_Blood Product']
    
    F = len(state_cols)
    A_dim = len(action_cols)
    
    # Initialize tensors
    X = torch.zeros((B, T, F), dtype=torch.float32)
    A = torch.zeros((B, T, A_dim), dtype=torch.float32)
    mask = torch.zeros((B, T), dtype=torch.float32)
    R = torch.zeros((B, T), dtype=torch.float32)
    
    # Normalize reward
    reward_mean = df_rl['reward_shaped'].mean()
    reward_std = df_rl['reward_shaped'].std()
    
    # Fill tensors
    for i, pid in enumerate(ids):
        df_pid = df_rl[df_rl['ID'] == pid].sort_values('Time')
        seq_len = min(len(df_pid), T)
        
        X[i, :seq_len, :] = torch.tensor(
            df_pid[state_cols].fillna(0).values[:seq_len], dtype=torch.float32
        )
        A[i, :seq_len, :] = torch.tensor(
            df_pid[action_cols].fillna(0).values[:seq_len], dtype=torch.float32
        )
        mask[i, :seq_len] = 1.0
        
        r = ((df_pid['reward_shaped'].values - reward_mean) / (reward_std + 1e-8))[:seq_len]
        R[i, :seq_len] = torch.tensor(r, dtype=torch.float32)
    
    # Split data
    train_idx, test_idx = train_test_split(np.arange(B), test_size=0.15, random_state=42)
    X_train, X_test = X[train_idx], X[test_idx]
    A_train, A_test = A[train_idx], A[test_idx]
    R_train, R_test = R[train_idx], R[test_idx]
    mask_train, mask_test = mask[train_idx], mask[test_idx]
    
    # Create dataloaders
    train_loader = DataLoader(
        TensorDataset(X_train, A_train, R_train, mask_train),
        batch_size=settings.BATCH_SIZE,
        shuffle=True
    )
    
    test_loader = DataLoader(
        TensorDataset(X_test, A_test, R_test, mask_test),
        batch_size=settings.BATCH_SIZE,
        shuffle=False
    )
    
    # Initialize model
    model = BiLSTMAttentionRL(F, A_dim).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=settings.LEARNING_RATE)
    criterion = nn.MSELoss(reduction='none')
    
    # Training loop
    train_losses, test_losses = [], []
    
    for epoch in range(settings.EPOCHS):
        # Training phase
        model.train()
        total_train_loss = 0.0
        
        for X_batch, A_batch, R_batch, mask_batch in train_loader:
            X_batch, A_batch, R_batch, mask_batch = [
                t.to(device) for t in (X_batch, A_batch, R_batch, mask_batch)
            ]
            
            optimizer.zero_grad()
            Q_pred, _ = model(X_batch)
            
            # Compute loss
            Q_taken = (Q_pred * A_batch).sum(dim=-1)
            Q_next = torch.zeros_like(Q_taken)
            Q_next[:, :-1] = Q_pred[:, 1:, :].max(dim=-1)[0].detach()
            target_Q = R_batch + settings.GAMMA * Q_next
            
            loss_all = criterion(Q_taken, target_Q)
            loss = (loss_all * mask_batch).sum() / mask_batch.sum()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_train_loss += loss.item() * X_batch.size(0)
        
        avg_train_loss = total_train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)
        
        # Testing phase
        model.eval()
        total_test_loss = 0.0
        
        with torch.no_grad():
            for X_batch, A_batch, R_batch, mask_batch in test_loader:
                X_batch, A_batch, R_batch, mask_batch = [
                    t.to(device) for t in (X_batch, A_batch, R_batch, mask_batch)
                ]
                
                Q_pred, _ = model(X_batch)
                
                # Compute loss
                Q_taken = (Q_pred * A_batch).sum(dim=-1)
                Q_next = torch.zeros_like(Q_taken)
                Q_next[:, :-1] = Q_pred[:, 1:, :].max(dim=-1)[0]
                target_Q = R_batch + settings.GAMMA * Q_next
                
                loss_all = criterion(Q_taken, target_Q)
                loss = (loss_all * mask_batch).sum() / mask_batch.sum()
                total_test_loss += loss.item() * X_batch.size(0)
        
        avg_test_loss = total_test_loss / len(test_loader.dataset)
        test_losses.append(avg_test_loss)
    
    # Create result object
    result = RLModelResult(
        name="BiLSTM-Attention-RL",
        train_loss_list=train_losses,
        test_loss_list=test_losses,
        Q_pred=Q_pred,
        A_batch=A_batch,
        R_batch=R_batch,
        mask_batch=mask_batch
    )
    
    return result